In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [7]:
i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.

In [8]:
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\bellpepper"

In [9]:
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    

Found 997 files belonging to 1 classes.
Using 798 files for training.


In [10]:
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set

Found 997 files belonging to 1 classes.
Using 199 files for validation.


In [11]:
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)

In [12]:
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())

Classes: ['Pepper__bell___Bacterial_spot']
Train batches: 25
Val batches: 4
Test batches: 3


In [13]:
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)

In [14]:
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [15]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers


In [16]:
num = len(class_names)


In [17]:
base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False

In [18]:
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint

In [19]:
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

In [20]:
x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)

In [21]:
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)

In [22]:
model = models.Model(inputs, outputs)

In [23]:
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [24]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]

In [25]:
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10


C:\Users\dedha\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\ops\nn.py:946: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 850ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00

25/25 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 24s 961ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - learning_rate: 0.0010
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 25s 993ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - learning_rate: 0.0010
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 25s 996ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - learning_rate: 3.0000e-04
